In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import linear_model
from sklearn.model_selection import train_test_split
import gc

In [ ]:
digits = pd.read_csv("input/svm/train.csv")
digits.info()

In [ ]:
digits.label.astype('category').value_counts()

100*(round(digits.label.astype('category').value_counts()/len(digits.index), 4))

digits.isnull().sum()

In [ ]:
x = digits.iloc[:, 1:]
y = digits.iloc[:, 0]

from sklearn.preprocessing import scale
x = scale(x)

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.10, random_state=101)

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
from sklearn import svm
from sklearn import metrics

svm_rbf = svm.SVC(kernel='rbf', probability=True)

svm_rbf.fit(x_train, y_train)

In [ ]:
predictions = svm_rbf.predict(x_test)
print(metrics.accuracy_score(y_true=y_test, y_pred=predictions))

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters = {'C': [1, 10, 100],
              'gamma': [1e-2, 1e-3, 1e-4]}

svc_grid_search = svm.SVC(kernel='rbf', probability=True)
clf = GridSearchCV(svc_grid_search, param_grid=parameters, scoring='accuracy', return_train_score=True)

clf.fit(x_train, y_train)

In [ ]:
cv_results = pd.DataFrame(clf.cv_results_)
cv_results

In [ ]:
cv_results['param_C'] = cv_results['param_C'].astype('int')

plt.figure(figsize=(16, 6))


plt.subplot(131)
gamma_01 = cv_results[cv_results['param_gamma']==0.01]

plt.plot(gamma_01['param_C'], gamma_01['mean_test_score'])
plt.plot(gamma_01['param_C'], gamma_01['mean_train_score'])
plt.xlabel('C')
plt.ylabel('Accuracy')
plt.title('Gamma=0.01')
plt.ylim([0.60, 1])
plt.legend(['test accuracy', 'train_accuracy'], loc='lower right')
plt.xscale('log')

plt.subplot(132)
gamma_001 = cv_results[cv_results['param_gamma'] == 0.001]

plt.plot(gamma_001['param_C'], gamma_001['mean_test_score'])
plt.plot(gamma_001['param_C'], gamma_001['mean_train_score'])
plt.xlabel('C')
plt.ylabel('Accuracy')
plt.title('Gamma=0.001')
plt.ylim([0.60, 1])
plt.legend(['test accuracy', 'train accuracy'], loc='lower right')
plt.xscale('log')


plt.subplot(133)
gamma_0001 = cv_results[cv_results['param_gamma']==0.0001]

plt.plot(gamma_0001['param_C'], gamma_0001['mean_test_score'])
plt.plot(gamma_0001['param_C'], gamma_0001['mean_train_score'])
plt.xlabel('C')
plt.ylabel('Accuracy')
plt.title('Gamma=0.0001')
plt.ylim([0.60, 1])
plt.legend(['test accuracy', 'train accuracy'], loc = 'lower right')
plt.xscale('log')


plt.show()

In [ ]:
best_C= 1
best_gamma = 0.001

svm_final = svm.SVC(kernel='rbf', C=best_C, gamma=best_gamma, probability=True)

svm_final.fit(x_train, y_train)

In [ ]:
predictions = svm_final.predict(x_test)

In [ ]:
confussion = metrics.confusion_matrix(y_true=y_test, y_pred=predictions)

test_accuracy = metrics.accuracy_score(y_true=y_test, y_pred=predictions)

print(test_accuracy, '\n')
print(confussion)

In [ ]:
import joblib

joblib.dump(svm_final, "mnist_svm_non_linear.pkl")